# 01. Generative vs. Discriminative Classifiers & The Ng-Jordan Crossover

### Historical Lineage & Theoretical Foundations
- **Lineage:** *Ng & Jordan (2001, On Discriminative vs. Generative Classifiers: A comparison of logistic regression and naive Bayes); Vapnik (1995, The Nature of Statistical Learning Theory); Fisher (1936, Linear Discriminant Analysis); Bishop (2006, Pattern Recognition and Machine Learning)*.
- **Core Purpose:** Formalizes the duality between generative classifiers modeling the joint distribution $p(\mathbf{x}, y) = p(y) p(\mathbf{x} \mid y)$ and discriminative classifiers modeling the class posterior $p(y \mid \mathbf{x})$ directly. Analyzes closed-form MLE moment matching vs. iterative gradient-based empirical risk minimization.

### Decision Boundary: When NOT to Use
> [!WARNING]
> - **DO NOT assume Gaussian class-conditionals when data contains multi-modal sub-clusters or heavy tails:** Generative GDA places class means at the geometric centroid, causing catastrophic classification failure when clusters are non-convex or multimodal; use discriminative neural classifiers.
> - **DO NOT train overparameterized discriminative models when samples are severely constrained ($N \ll D$):** Discriminative models overfit rapidly with sample complexity $\mathcal{O}(D)$; generative models converge to their asymptotic error with $\mathcal{O}(\log D)$ samples.
> - **DO NOT rely on discriminative softmax for out-of-distribution (OOD) detection:** Discriminative models output overconfident probabilities far from the training support; only generative models provide marginal density $p(\mathbf{x})$ for anomaly isolation.


### 2. Mathematical Foundations & Governing Equations

#### 1. Generative Gaussian Discriminant Analysis (GDA)
$$p(y = c) = \pi_c, \quad p(\mathbf{x} \mid y = c) = \frac{1}{(2\pi)^{D/2} |\mathbf{\Sigma}|^{1/2}} \exp\left( -\frac{1}{2} (\mathbf{x} - \mathbf{\mu}_c)^\top \mathbf{\Sigma}^{-1} (\mathbf{x} - \mathbf{\mu}_c) \right)$$

#### 2. Bayes Posterior Class Duality & Derived Affine Logits
$$p(y = c \mid \mathbf{x}) = \frac{\exp(\mathbf{x}^\top \mathbf{w}_c + b_c)}{\sum_{k=0}^{C-1} \exp(\mathbf{x}^\top \mathbf{w}_k + b_k)}$$
$$\mathbf{w}_c = \mathbf{\Sigma}^{-1} \mathbf{\mu}_c, \quad b_c = -\frac{1}{2} \mathbf{\mu}_c^\top \mathbf{\Sigma}^{-1} \mathbf{\mu}_c + \ln \pi_c$$

#### 3. Discriminative Cross-Entropy Objective
$$\mathcal{L}_{\text{CE}}(\mathbf{W}, \mathbf{b}) = -\frac{1}{N} \sum_{i=1}^N \sum_{c=0}^{C-1} y_{ic} \ln p(y = c \mid \mathbf{x}_i; \mathbf{W}, \mathbf{b})$$

**Asymptotic Complexity Analysis:**
- **Generative GDA Parameter Fitting:** Time $\mathcal{O}(N D + D^3)$ single-pass moment calculation and covariance inversion; Space $\mathcal{O}(D^2)$ covariance storage.
- **Discriminative Softmax Regression:** Time $\mathcal{O}(T N D C)$ across $T$ gradient iterations; Space $\mathcal{O}(D C)$ parameter weights.
- **Ng-Jordan Sample Complexity:** GDA approaches asymptotic error at rate $\mathcal{O}(\log D / N)$; Discriminative regression converges at rate $\mathcal{O}(D / N)$ with lower asymptotic bias.


In [ ]:
from pathlib import Path

import torch
import torch.nn as nn

import core

env = core.setup_env(seed=42)
ckpt_path = Path("weights/01_paradigms_and_dichotomies/01_generative_vs_discriminative_classifiers.safetensors")
if not ckpt_path.is_file():
    ckpt_path = Path("../../weights/01_paradigms_and_dichotomies/01_generative_vs_discriminative_classifiers.safetensors")
weights, test_in, golden_out, meta = core.parity.load_reference_checkpoint(ckpt_path)

batch_size = int(meta.get("batch_size", 8))
in_dim = int(meta.get("in_dim", 6))
num_classes = int(meta.get("num_classes", 3))

print(f"Device: {env.device} | Batch: {batch_size} | Dim: {in_dim} | Classes: {num_classes} | Checkpoint: {ckpt_path.name}")


In [ ]:
def fit_gda_parameters(
    x: torch.Tensor, y: torch.Tensor, num_classes: int, reg_lambda: float = 1e-4
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    num_samples, dim = x.shape
    means = torch.zeros(num_classes, dim, dtype=x.dtype, device=x.device)
    priors = torch.zeros(num_classes, dtype=x.dtype, device=x.device)
    for c in range(num_classes):
        mask = y == c
        priors[c] = float(mask.sum().item()) / float(num_samples)
        means[c] = x[mask].mean(dim=0)

    centered_x = x.clone()
    for c in range(num_classes):
        mask = y == c
        centered_x[mask] = x[mask] - means[c]

    shared_cov = (centered_x.T @ centered_x) / float(num_samples)
    reg_cov = shared_cov + reg_lambda * torch.eye(dim, dtype=x.dtype, device=x.device)
    cov_inv = torch.linalg.inv(reg_cov)

    w_derived = torch.zeros(dim, num_classes, dtype=x.dtype, device=x.device)
    b_derived = torch.zeros(num_classes, dtype=x.dtype, device=x.device)
    for c in range(num_classes):
        mu_c = means[c]
        sigma_inv_mu = cov_inv @ mu_c
        w_derived[:, c] = sigma_inv_mu
        b_derived[c] = -0.5 * torch.dot(mu_c, sigma_inv_mu) + torch.log(priors[c] + 1e-12)

    return means, reg_cov, priors, w_derived, b_derived

class GaussianDiscriminantAnalysis:
    def __init__(self, w_derived: torch.Tensor, b_derived: torch.Tensor) -> None:
        self.w_derived = w_derived
        self.b_derived = b_derived

    def predict_logits(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.w_derived + self.b_derived

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.predict_logits(x), dim=-1)

class DiscriminativeClassifier(nn.Module):
    def __init__(self, in_features: int, num_classes: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, num_classes))
        self.bias = nn.Parameter(torch.empty(num_classes))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

    def predict_proba(self, x: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.forward(x), dim=-1)

disc_model = DiscriminativeClassifier(in_dim, num_classes)
with torch.no_grad():
    disc_model.weight.copy_(weights["weight"])
    disc_model.bias.copy_(weights["bias"])
gda_model = GaussianDiscriminantAnalysis(weights["gda_weight"], weights["gda_bias"])
print("DiscriminativeClassifier and GaussianDiscriminantAnalysis initialized.")


In [ ]:
# 1. Discriminative Forward Pass Parity
disc_pred = disc_model(test_in)
torch.testing.assert_close(disc_pred, golden_out, atol=1e-5, rtol=1e-5)

# 2. Generative GDA Derived Logits Parity
gda_pred = gda_model.predict_logits(test_in)
torch.testing.assert_close(gda_pred, weights["gda_logits"], atol=1e-5, rtol=1e-5)

# 3. Probability Simplex Axiom Check: sum_c p(y=c | x) == 1.0
probs = disc_model.predict_proba(test_in)
torch.testing.assert_close(probs.sum(dim=-1), torch.ones(batch_size), atol=1e-5, rtol=1e-5)
assert (probs >= 0.0).all() and (probs <= 1.0).all(), "Probabilities must lie strictly in [0, 1]"
print("Transitive Parity & Probability Simplex Axioms verified successfully.")


In [ ]:
def run_discriminative():
    return disc_model(test_in)

def run_gda():
    return gda_model.predict_logits(test_in)

def run_softmax():
    return disc_model.predict_proba(test_in)

track_fns = {"Discriminative Fwd": run_discriminative, "Generative GDA Logits": run_gda, "Softmax Probabilities": run_softmax}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=batch_size)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# [!WARNING] PATHOLOGICAL MISUSE & FAILURE MODE: Model Misspecification on Non-Gaussian Multi-Modal Data
# 1. Flawed Anti-Pattern: Fitting GDA on non-Gaussian data (Class 0 has two sub-clusters at x = -3 and x = +3)
gen_fail = torch.Generator().manual_seed(123)
n_pts = 100
# Class 0: bimodal mixture at -3.0 and +3.0
c0_part1 = torch.randn(n_pts // 2, 2, generator=gen_fail) + torch.tensor([-3.0, 0.0])
c0_part2 = torch.randn(n_pts // 2, 2, generator=gen_fail) + torch.tensor([3.0, 0.0])
x_c0 = torch.cat([c0_part1, c0_part2], dim=0)
# Class 1: unimodal at 0.0
x_c1 = torch.randn(n_pts, 2, generator=gen_fail) + torch.tensor([0.0, 0.0])

x_synthetic = torch.cat([x_c0, x_c1], dim=0)
y_synthetic = torch.cat([torch.zeros(n_pts, dtype=torch.long), torch.ones(n_pts, dtype=torch.long)])

# Fit GDA: Class 0 mean collapses to 0.0 (the center of the two clusters, overlapping Class 1!)
_, _, _, w_fail, b_fail = fit_gda_parameters(x_synthetic, y_synthetic, num_classes=2)
gda_logits = x_synthetic @ w_fail + b_fail
gda_preds = torch.argmax(gda_logits, dim=-1)
gda_err_rate = float((gda_preds != y_synthetic).float().mean().item())

assert gda_err_rate > 0.35, f"GDA error rate {gda_err_rate:.2%} should exceed 35% on non-Gaussian mixture"
print(f"[FAILURE DEMONSTRATED] Generative GDA suffered {gda_err_rate:.2%} error due to Gaussian centroid assumption collapse!")

# 2. Correct Pattern: Discriminative logistic regression trained directly on posterior
lin = nn.Linear(2, 2)
opt = torch.optim.SGD(lin.parameters(), lr=0.1)
crit = nn.CrossEntropyLoss()
for _ in range(50):
    opt.zero_grad()
    loss = crit(lin(x_synthetic), y_synthetic)
    loss.backward()
    opt.step()
print("[CORRECT] Discriminative classifier trained directly without assuming unimodal Gaussian marginals.")

# 3. Compilation Verification
ok_exp, msg_exp = core.export.verify_torch_export(disc_model, test_in)
ok_comp, msg_comp = core.export.verify_torch_compile_fullgraph(disc_model, test_in)
assert ok_exp and ok_comp, "Compilation / export failed"
print("[CORRECT] Discriminative model verified with torch.compile(fullgraph=True) and exported.")


## Architectural Topology, Key Takeaways & Curriculum Navigation

### Generative vs. Discriminative Classifier Topology
<svg width="680" height="180" xmlns="http://www.w3.org/2000/svg" style="background:#0f172a; border-radius:8px; font-family:system-ui, sans-serif;">
  <defs>
    <marker id="arrow-s" viewBox="0 0 10 10" refX="6" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse">
      <path d="M 0 1 L 10 5 L 0 9 z" fill="#38bdf8"/>
    </marker>
  </defs>
  <text x="30" y="25" fill="#94a3b8" font-size="12" font-weight="bold">GENERATIVE VS. DISCRIMINATIVE CLASSIFIER PARADIGM</text>
  <rect x="30" y="50" width="150" height="35" rx="6" fill="#1e293b" stroke="#38bdf8" stroke-width="1.5"/>
  <text x="105" y="72" fill="#38bdf8" font-size="11" text-anchor="middle">Samples (X, y)</text>
  <line x1="180" y1="67" x2="240" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="240" y="50" width="190" height="35" rx="6" fill="#1e293b" stroke="#10b981" stroke-width="1.5"/>
  <text x="335" y="72" fill="#10b981" font-size="11" text-anchor="middle">GDA: p(x, y) = p(y)p(x|y)</text>
  <line x1="430" y1="67" x2="490" y2="67" stroke="#38bdf8" stroke-width="2" marker-end="url(#arrow-s)"/>
  <rect x="490" y="50" width="160" height="35" rx="6" fill="#1e293b" stroke="#a855f7" stroke-width="1.5"/>
  <text x="570" y="72" fill="#a855f7" font-size="11" text-anchor="middle">Derived Linear Logits</text>
  <rect x="30" y="110" width="620" height="40" rx="6" fill="#0f172a" stroke="#f59e0b" stroke-width="1.5" stroke-dasharray="4 4"/>
  <text x="340" y="135" fill="#f59e0b" font-size="11" text-anchor="middle">Ng-Jordan Crossover: GDA wins at small N (O(log D)), Discriminative wins as N -&gt; inf</text>
</svg>

### Key Takeaways
1. **Mathematical Duality:** Under Gaussian class-conditionals with shared covariance, GDA induces the exact same linear functional form $\mathbf{x}^\top \mathbf{w} + b$ as logistic regression.
2. **The Ng-Jordan Crossover:** Generative classifiers reach their asymptotic error regime with fewer samples ($N \sim \mathcal{O}(\log D)$), but have higher asymptotic error when the generative model is misspecified.
3. **Vapnik's Imperative:** When sample size is sufficient ($N \gg D$), discriminative modeling directly optimizes conditional posterior likelihood without wasting capacity on the input marginal $p(\mathbf{x})$.

---
**Curriculum Navigation:**  
&larr; [Previous: 00. Foundations / 11. Randomized Numerical Linear Algebra, Spectral Autograd & Hutch++](../00_foundations/11_randomized_numerical_linear_algebra_spectral_autograd_and_hutchpp.ipynb) | [Master Index: README.md](../../README.md) | [Next: 02. Tree vs. Kernel vs. Neural Inductive Biases](02_tree_vs_kernel_vs_neural_inductive_biases.ipynb) &rarr;
